# Practical Guide to Support Vector Machines in Python

Complete maintained companion: Iris classification, moons polynomial/RBF kernels and seeded linear/quadratic regression. Python 3.11+; CPU; no API keys or external dataset. Run top to bottom. Training scores are not test metrics. The original source appendix records unchanged code and immutable gist revisions.

Maintenance: hinge LinearSVC requires dual=True; helper definition moved before first use; absent degree-10/RBF comparisons supplied; LinearSVR residual masks are plotting diagnostics, not optimizer-certified support indices. Source prose guidance about dual=False with hinge is invalid. Increasing C weakens regularization, whereas gamma controls influence radius. General speed/complexity and epsilon-insensitivity claims have limits. These findings are documented in the website import review.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==2.3.3', 'scipy==1.16.2', 'scikit-learn==1.7.2',
                'matplotlib==3.10.9', 'joblib==1.5.2', 'threadpoolctl==3.6.0'], check=True)
import warnings
import numpy as np
import scipy
import sklearn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVC, LinearSVR, SVR
from sklearn.exceptions import ConvergenceWarning
from sklearn.base import clone
from threadpoolctl import threadpool_limits
%matplotlib inline
warnings.filterwarnings('error', category=ConvergenceWarning)
thread_limits = threadpool_limits(limits=1)
np.set_printoptions(precision=8, linewidth=100)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
models = {}
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__, '| scikit-learn:', sklearn.__version__)
print('CPU; bundled Iris and seeded synthetic data; no API keys or data downloads.')

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
print('SVM imports ready.')

In [ ]:
iris = datasets.load_iris()
X = iris["data"][:, (2, 3)] # petal length, petal width
y = (iris["target"] == 2).astype(np.float64) # Iris-Virginica
X_iris, y_iris = X.copy(), y.copy()
print('Iris shape:', X.shape, '| Binary counts:', np.bincount(y.astype(int)).tolist())

In [ ]:
# defining the scaler and the SVM classifer 

svm_clf = Pipeline([
("scaler", StandardScaler()),
("linear_svc", LinearSVC(C=1, loss="hinge", dual=True, random_state=42, max_iter=100000)),
])

svm_clf.fit(X, y)
models['iris'] = (svm_clf, X.copy(), y.copy())
print('Iris training accuracy:', round(svm_clf.score(X, y), 6))
print('Predictions:', svm_clf.predict([[5.5, 1.7], [1.5, 0.3]]).astype(int).tolist())

In [ ]:
# create moon data using make_moons function 
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

# Create sklearn pipeline that first create polynomial features 
#then scale them and create linearSVM classifer
clf = Pipeline([
("poly_features", PolynomialFeatures(degree=3)),
("scaler", StandardScaler()),
("svm_clf", LinearSVC(C=10, loss="hinge", dual=True, random_state=42, max_iter=1000000))
])

clf.fit(X, y)

"""
# fit the model
clf = svm.SVC(kernel='linear')
clf.fit(X, y)
"""

# plot the decision function for each datapoint on the grid
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.figure(figsize=(10,10))
plt.imshow(
    Z,
    interpolation="nearest",
    extent=(xx.min(), xx.max(), yy.min(), yy.max()),
    aspect="auto",
    origin="lower",
    cmap=plt.cm.PuOr_r,
)
contours = plt.contour(xx, yy, Z, levels=[0], linewidths=2, linestyles="dashed")
plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired, edgecolors="k")
plt.axis([-3, 3, -3, 3])
plt.title('Linear SVM classifier using polynomial features')
plt.xlabel('X1')
plt.ylabel('X2')
plt.show()
X_moons, y_moons = X.copy(), y.copy()
models['explicit-poly'] = (clf, X.copy(), y.copy())
print('Degree-3 feature count:', clf[0].n_output_features_)
print('Moons training accuracy:', round(clf.score(X, y), 6))

In [ ]:
# plot the decision function for each datapoint on the grid
def decision_boundary_plotting(xx, yy,poly_kernel_svm_clf ,title):
    Z = poly_kernel_svm_clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(10,10))
    plt.imshow(
        Z,
        interpolation="nearest",
        extent=(xx.min(), xx.max(), yy.min(), yy.max()),
        aspect="auto",
        origin="lower",
        cmap=plt.cm.PuOr_r,
    )

    contours = plt.contour(xx, yy, Z, levels=[0], linewidths=2, linestyles="dashed")
    plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired, edgecolors="k")
    plt.axis([-3, 3, -3, 3])
    plt.title(title)
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.show()
print('Decision-boundary helper ready.')

In [ ]:
from sklearn.svm import SVC
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

poly_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5))
    ])
poly_kernel_svm_clf.fit(X, y)
decision_boundary_plotting(xx, yy,poly_kernel_svm_clf ,'SVM classifier using polynomial features d=3, r=1,c=5')
models['poly3'] = (poly_kernel_svm_clf, X.copy(), y.copy())
print('Degree-3 training accuracy:', round(poly_kernel_svm_clf.score(X, y), 6))

In [ ]:
poly10 = Pipeline([('scaler', StandardScaler()),
                   ('svm_clf', SVC(kernel='poly', degree=10, coef0=10, C=5))])
poly10.fit(X, y)
models['poly10'] = (poly10, X.copy(), y.copy())
decision_boundary_plotting(xx, yy, poly10, 'Polynomial kernel: degree=10, coef0=10, C=5')
print('Degree-10 training accuracy:', round(poly10.score(X, y), 6))

In [ ]:
landmarks = np.array([-2.0, 1.0])
point = -1.0
similarities = np.exp(-0.3 * (point - landmarks)**2)
print('RBF similarities at x=-1:', similarities)

In [ ]:
from sklearn.svm import SVC
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

rbf_kernel_svm_clf = Pipeline([
("scaler", StandardScaler()),
("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001))
])
rbf_kernel_svm_clf.fit(X, y)
decision_boundary_plotting(xx, yy,rbf_kernel_svm_clf ,'SVM classifier using RBF Kernel features γ=5, C=0.001')
models['rbf5-small'] = (rbf_kernel_svm_clf, X.copy(), y.copy())
print('RBF gamma=5, C=0.001 training accuracy:', round(rbf_kernel_svm_clf.score(X, y), 6))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (gamma, penalty) in zip(axes, [(5, 1000), (0.1, 0.001), (0.1, 1000)]):
    model = Pipeline([('scaler', StandardScaler()),
                      ('svm_clf', SVC(kernel='rbf', gamma=gamma, C=penalty))])
    model.fit(X, y)
    models[f'rbf-{gamma}-{penalty}'] = (model, X.copy(), y.copy())
    decisions = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.imshow(decisions, extent=(-3, 3, -3, 3), origin='lower', aspect='auto', cmap=plt.cm.PuOr_r)
    ax.contour(xx, yy, decisions, levels=[0], colors='k', linestyles='dashed')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, edgecolors='k', s=20)
    ax.set(xlim=(-3, 3), ylim=(-3, 3), xlabel='X1', title=f'gamma={gamma}, C={penalty}')
    print(f'gamma={gamma}, C={penalty}: training accuracy={model.score(X, y):.6f}')
axes[0].set_ylabel('X2')
plt.tight_layout()
plt.show()

In [ ]:
# Lets first create random data
np.random.seed(42)
m = 50
X = 2 * np.random.rand(m, 1)
y = (4 + 3 * X + np.random.randn(m, 1)).ravel()
X_linear, y_linear = X.copy(), y.copy()
print('Linear regression shape:', X.shape, '| First targets:', y[:3])

In [ ]:
svm_reg1 = LinearSVR(epsilon=1.5, random_state=42, dual=True, max_iter=100000) # svm with ϵ = 1.5
svm_reg2 = LinearSVR(epsilon=0.5, random_state=42, dual=True, max_iter=100000) # svm with ϵ = 0.5
svm_reg1.fit(X, y) # fitting SVM 1
svm_reg2.fit(X, y) # fitting SVM 2

def find_support_vectors(svm_reg, X, y):
    y_pred = svm_reg.predict(X)
    off_margin = (np.abs(y - y_pred) >= svm_reg.epsilon)
    return np.flatnonzero(off_margin)

svm_reg1.support_ = find_support_vectors(svm_reg1, X, y)
svm_reg2.support_ = find_support_vectors(svm_reg2, X, y)

eps_x1 = 1
eps_y_pred = float(svm_reg1.predict([[eps_x1]])[0])
for model in [svm_reg1, svm_reg2]:
    print(f'epsilon={model.epsilon}: off-tube points={len(model.support_)}, training MSE={np.mean((y-model.predict(X))**2):.8f}')

In [ ]:
def plot_svm_regression(svm_reg, X, y, axes):
    x1s = np.linspace(axes[0], axes[1], 100).reshape(100, 1)
    y_pred = svm_reg.predict(x1s)
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$")
    plt.plot(x1s, y_pred + svm_reg.epsilon, "k--")
    plt.plot(x1s, y_pred - svm_reg.epsilon, "k--")
    plt.scatter(X[svm_reg.support_], y[svm_reg.support_], s=180, facecolors='#FFAAAA')
    plt.plot(X, y, "bo")
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.legend(loc="upper left", fontsize=18)
    plt.axis(axes)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_reg1, X, y, [0, 2, 3, 11])
plt.title(r"$\epsilon = {}$".format(svm_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
#plt.plot([eps_x1, eps_x1], [eps_y_pred, eps_y_pred - svm_reg1.epsilon], "k-", linewidth=2)
plt.annotate(
        '', xy=(eps_x1, eps_y_pred), xycoords='data',
        xytext=(eps_x1, eps_y_pred - svm_reg1.epsilon),
        textcoords='data', arrowprops={'arrowstyle': '<->', 'linewidth': 1.5}
    )

plt.text(0.91, 5.6, r"$\epsilon$", fontsize=20)
plt.sca(axes[1])
plot_svm_regression(svm_reg2, X, y, [0, 2, 3, 11])
plt.title(r"$\epsilon = {}$".format(svm_reg2.epsilon), fontsize=18)
plt.show()
print('Both linear epsilon-tube plots rendered.')

In [ ]:
np.random.seed(42)
m = 100
X = 2 * np.random.rand(m, 1) - 1
y = (0.2 + 0.1 * X + 0.5 * X**2 + np.random.randn(m, 1)/10).ravel()
X_quad, y_quad = X.copy(), y.copy()
print('Quadratic regression shape:', X.shape, '| First targets:', y[:3])

In [ ]:
from sklearn.svm import SVR

svm_poly_reg1 = SVR(kernel="poly", degree=2, C=100, epsilon=0.1, gamma="scale")
svm_poly_reg2 = SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1, gamma="scale")
svm_poly_reg1.fit(X, y)
svm_poly_reg2.fit(X, y)
for model in [svm_poly_reg1, svm_poly_reg2]:
    print(f'C={model.C}: support vectors={len(model.support_)}, training MSE={np.mean((y-model.predict(X))**2):.8f}')

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_poly_reg1, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg1.degree, svm_poly_reg1.C, svm_poly_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
plt.sca(axes[1])
plot_svm_regression(svm_poly_reg2, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg2.degree, svm_poly_reg2.C, svm_poly_reg2.epsilon), fontsize=18)
plt.show()
print('Both polynomial regression plots rendered.')

In [ ]:
from sklearn.svm import SVR
svm_poly_reg = SVR(kernel="poly", degree=2, C=100, epsilon=0.1)
svm_poly_reg.fit(X, y)
plt.figure(figsize=(7, 4))
plot_svm_regression(svm_poly_reg, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg.degree, svm_poly_reg.C, svm_poly_reg.epsilon), fontsize=18)
plt.tight_layout()
plt.show()
print('Standalone polynomial SVR training MSE:', round(np.mean((y-svm_poly_reg.predict(X))**2), 8))

In [ ]:
# Independent mathematical checks, not a generalization benchmark.
from numpy.testing import assert_allclose, assert_array_equal
assert X_iris.shape == (150, 2) and X_moons.shape == (100, 2)
assert X_linear.shape == (50, 1) and X_quad.shape == (100, 1)
assert_allclose(similarities, [np.exp(-0.3), np.exp(-1.2)], rtol=0, atol=1e-14)
kernel_checks = 0
for name, (pipe, inputs, targets) in models.items():
    transformed = inputs
    for stage in pipe[:-1]:
        transformed = stage.transform(transformed)
    final = pipe[-1]
    if isinstance(final, LinearSVC):
        manual = transformed @ final.coef_.T + final.intercept_
        manual = manual.ravel()
        assert final.dual is True and final.n_iter_ < final.max_iter
    else:
        support = final.support_vectors_
        dot = transformed @ support.T
        if final.kernel == 'poly':
            kernel = (final._gamma * dot + final.coef0)**final.degree
        else:
            d2 = ((transformed[:, None, :] - support[None, :, :])**2).sum(axis=2)
            kernel = np.exp(-final._gamma * d2)
        manual = (kernel @ final.dual_coef_.T).ravel() + final.intercept_[0]
        kernel_checks += len(inputs)
    assert_allclose(manual, pipe.decision_function(inputs), rtol=1e-9, atol=1e-7)
    assert_array_equal((manual > 0).astype(int), pipe.predict(inputs))
    assert_allclose(clone(pipe).fit(inputs, targets).decision_function(inputs),
                    pipe.decision_function(inputs), rtol=1e-10, atol=1e-10)
for reg in [svm_reg1, svm_reg2]:
    pred = X_linear @ reg.coef_ + reg.intercept_[0]
    assert_allclose(pred, reg.predict(X_linear), atol=1e-12)
    assert_array_equal(reg.support_, np.flatnonzero(np.abs(y_linear-pred) >= reg.epsilon))
    assert reg.n_iter_ < reg.max_iter
for reg in [svm_poly_reg1, svm_poly_reg2, svm_poly_reg]:
    kernel = (reg._gamma * (X_quad @ reg.support_vectors_.T) + reg.coef0)**reg.degree
    manual = (kernel @ reg.dual_coef_.T).ravel() + reg.intercept_[0]
    assert_allclose(manual, reg.predict(X_quad), atol=1e-10)
    assert_allclose(clone(reg).fit(X_quad, y_quad).predict(X_quad), manual, atol=1e-10)
# Probe non-training locations; duplicate and permutation batch consistency.
probe = np.array([[-1.1, 0.2], [0.4, -0.6], [2.0, 1.2]])
for name, (model, inputs, _) in models.items():
    assert_allclose(model.decision_function(inputs[::-1])[::-1], model.decision_function(inputs))
    assert_array_equal(model.predict(np.repeat(probe, 2, axis=0)), np.repeat(model.predict(probe), 2))
assert_allclose(svm_poly_reg.predict(X_quad), svm_poly_reg1.predict(X_quad), atol=1e-12)
assert all(np.isfinite(model.decision_function(inputs)).all() for model, inputs, _ in models.values())
print('All fourteen recovered gist examples and three missing comparison examples passed.')
print('Independent linear/poly/RBF decision reconstruction:', len(models), 'classifiers;', kernel_checks, 'kernel scores.')
print('All five regression models match independent coefficient/kernel predictions.')
print('Seeded refits, label/prediction batch consistency, epsilon masks and RBF landmark values pass.')
print('Training scores describe these teaching datasets, not held-out performance.')
thread_limits.restore_original_limits()

## Complete original gist snapshots (not executed)

### Original 1: 0282c92db6c4da230d6a6ea0e1f134b1
Revision: `35524d1c1d4779967ec9f51bb84b8f06481f8812`

```python
import numpy as np
from sklearn import datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
```

### Original 2: f4edcc2773c4f31e66497be8904d3ac8
Revision: `f6ef1c64557e4d3937cae12c5eee3e75e45716f8`

```python
iris = datasets.load_iris()
X = iris["data"][:, (2, 3)] # petal length, petal width
y = (iris["target"] == 2).astype(np.float64) # Iris-Virginica
```

### Original 3: 1653e4e947110d7f81fbcdaef43cdfa8
Revision: `74ee79e5dfb74fa8e61cc259a30cf611e19cd391`

```python
# defining the scaler and the SVM classifer 

svm_clf = Pipeline([
("scaler", StandardScaler()),
("linear_svc", LinearSVC(C=1, loss="hinge")),
])

svm_clf.fit(X, y)
```

### Original 4: 76b63f7acc27be1a1d953414c0f852cf
Revision: `460f0461935efb3ab6b86e2954b8904e15bbb688`

```python
# create moon data using make_moons function 
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

# Create sklearn pipeline that first create polynomial features 
#then scale them and create linearSVM classifer
clf = Pipeline([
("poly_features", PolynomialFeatures(degree=3)),
("scaler", StandardScaler()),
("svm_clf", LinearSVC(C=10, loss="hinge"))
])

clf.fit(X, y)

"""
# fit the model
clf = svm.SVC(kernel='linear')
clf.fit(X, y)
"""

# plot the decision function for each datapoint on the grid
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.figure(figsize=(10,10))
plt.imshow(
    Z,
    interpolation="nearest",
    extent=(xx.min(), xx.max(), yy.min(), yy.max()),
    aspect="auto",
    origin="lower",
    cmap=plt.cm.PuOr_r,
)
contours = plt.contour(xx, yy, Z, levels=[0], linewidths=2, linestyles="dashed")
plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired, edgecolors="k")
plt.axis([-3, 3, -3, 3])
plt.title('Linear SVM classifier using polynomial features')
plt.xlabel('X1')
plt.ylabel('X2')
plt.show()
```

### Original 5: 71d4ea974d0815c865ce3249a3a69747
Revision: `ad2eaf2138981ccf2fb427cbd30afb6011ebf120`

```python
from sklearn.svm import SVC
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

poly_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5))
    ])
poly_kernel_svm_clf.fit(X, y)
decision_boundary_plotting(xx, yy,poly_kernel_svm_clf ,'SVM classifier using polynomial features d=3, r=1,c=5')
```

### Original 6: ac9f67fb0b952698299868fd2330b632
Revision: `4c8c403f3feb0b458896943a28059436f2642dda`

```python
# plot the decision function for each datapoint on the grid
def decision_boundary_plotting(xx, yy,poly_kernel_svm_clf ,title):
    Z = poly_kernel_svm_clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(10,10))
    plt.imshow(
        Z,
        interpolation="nearest",
        extent=(xx.min(), xx.max(), yy.min(), yy.max()),
        aspect="auto",
        origin="lower",
        cmap=plt.cm.PuOr_r,
    )

    contours = plt.contour(xx, yy, Z, levels=[0], linewidths=2, linestyles="dashed")
    plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired, edgecolors="k")
    plt.axis([-3, 3, -3, 3])
    plt.title(title)
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.show()
```

### Original 7: cfba193c5cf4ac9c224d09b1e9cc5320
Revision: `7291bcac571625b4156ea9f4eace13e38f45e607`

```python
from sklearn.svm import SVC
X, y =  make_moons(noise=0.3, random_state=0)

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
np.random.seed(0)

rbf_kernel_svm_clf = Pipeline([
("scaler", StandardScaler()),
("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001))
])
rbf_kernel_svm_clf.fit(X, y)
decision_boundary_plotting(xx, yy,rbf_kernel_svm_clf ,'SVM classifier using RBF Kernel features γ=5, C=0.001')
```

### Original 8: b013a47342da1744b158b5d6e195a554
Revision: `28c23f52bf4c2e0158e290877b9e93688bd9e255`

```python
# Lets first create random data
np.random.seed(42)
m = 50
X = 2 * np.random.rand(m, 1)
y = (4 + 3 * X + np.random.randn(m, 1)).ravel()
```

### Original 9: c713f1739cd766b2a2dce30a27d9a42d
Revision: `ab01a3c31140fa52184889b1d42d390d102ba59e`

```python
svm_reg1 = LinearSVR(epsilon=1.5, random_state=42) # svm with ϵ = 1.5
svm_reg2 = LinearSVR(epsilon=0.5, random_state=42) # svm with ϵ = 0.5
svm_reg1.fit(X, y) # fitting SVM 1
svm_reg2.fit(X, y) # fitting SVM 2

def find_support_vectors(svm_reg, X, y):
    y_pred = svm_reg.predict(X)
    off_margin = (np.abs(y - y_pred) >= svm_reg.epsilon)
    return np.argwhere(off_margin)

svm_reg1.support_ = find_support_vectors(svm_reg1, X, y)
svm_reg2.support_ = find_support_vectors(svm_reg2, X, y)

eps_x1 = 1
eps_y_pred = svm_reg1.predict([[eps_x1]])
```

### Original 10: 0f18620ec7a69418aad278f1191c4581
Revision: `c51b88261ce470f2a0fd84031629b636fccff0f2`

```python
def plot_svm_regression(svm_reg, X, y, axes):
    x1s = np.linspace(axes[0], axes[1], 100).reshape(100, 1)
    y_pred = svm_reg.predict(x1s)
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$")
    plt.plot(x1s, y_pred + svm_reg.epsilon, "k--")
    plt.plot(x1s, y_pred - svm_reg.epsilon, "k--")
    plt.scatter(X[svm_reg.support_], y[svm_reg.support_], s=180, facecolors='#FFAAAA')
    plt.plot(X, y, "bo")
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.legend(loc="upper left", fontsize=18)
    plt.axis(axes)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_reg1, X, y, [0, 2, 3, 11])
plt.title(r"$\epsilon = {}$".format(svm_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
#plt.plot([eps_x1, eps_x1], [eps_y_pred, eps_y_pred - svm_reg1.epsilon], "k-", linewidth=2)
plt.annotate(
        '', xy=(eps_x1, eps_y_pred), xycoords='data',
        xytext=(eps_x1, eps_y_pred - svm_reg1.epsilon),
        textcoords='data', arrowprops={'arrowstyle': '<->', 'linewidth': 1.5}
    )

plt.text(0.91, 5.6, r"$\epsilon$", fontsize=20)
plt.sca(axes[1])
plot_svm_regression(svm_reg2, X, y, [0, 2, 3, 11])
plt.title(r"$\epsilon = {}$".format(svm_reg2.epsilon), fontsize=18)
plt.show()
```

### Original 11: f3cc1a88e8b05068d88de9400b893c20
Revision: `53ecf179e84b98f6f84dd80fb62ed8683ccfc13b`

```python
np.random.seed(42)
m = 100
X = 2 * np.random.rand(m, 1) - 1
y = (0.2 + 0.1 * X + 0.5 * X**2 + np.random.randn(m, 1)/10).ravel()
```

### Original 12: 113cbd67feb69d3cc640153cece557de
Revision: `fe1aea95f57a0223b6ffef9580e7241a1124e2da`

```python
from sklearn.svm import SVR

svm_poly_reg1 = SVR(kernel="poly", degree=2, C=100, epsilon=0.1, gamma="scale")
svm_poly_reg2 = SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1, gamma="scale")
svm_poly_reg1.fit(X, y)
svm_poly_reg2.fit(X, y)
```

### Original 13: 94a129acd0fd40b2d314ec16e66012f7
Revision: `9a78c77622593dc103f4ee91df1ee3cbb790bbb7`

```python
fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_poly_reg1, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg1.degree, svm_poly_reg1.C, svm_poly_reg1.epsilon), fontsize=18)
plt.ylabel(r"$y$", fontsize=18, rotation=0)
plt.sca(axes[1])
plot_svm_regression(svm_poly_reg2, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg2.degree, svm_poly_reg2.C, svm_poly_reg2.epsilon), fontsize=18)
plt.show()
```

### Original 14: 6f0ce4e7ecfaabe97f5c558041975a63
Revision: `afee2b8610fd82a9bbce72376ad2644f63722861`

```python
from sklearn.svm import SVR
svm_poly_reg = SVR(kernel="poly", degree=2, C=100, epsilon=0.1)
svm_poly_reg.fit(X, y)
plot_svm_regression(svm_poly_reg, X, y, [-1, 1, 0, 1])
plt.title(r"$degree={}, C={}, \epsilon = {}$".format(svm_poly_reg2.degree, svm_poly_reg2.C, svm_poly_reg2.epsilon), fontsize=18)
```

